# Steerable Pyramid

Used https://pyrtools.readthedocs.io/en/latest/ for steerabe pyramid. not my own converted matlab functions

**Core functions**

In [6]:
import numpy as np
from pyrtools.pyramids import SteerablePyramidFreq

def norm_energies(pyr_coeffs, num_orientations, sigma=0.1):
    """
    Compute contrast-normalized local energy responses from a steerable pyramid.

    Parameters:
        pyr_coeffs (dict): Coefficients of the steerable pyramid.
        num_orientations (int): Number of orientation subbands at each scale.
        sigma (float): Semi-saturation constant for contrast normalization.

    Returns:
        dict: Normalized coefficients.
    """
    # Compute energy (magnitude squared)
    energies = {k: np.abs(v)**2 for k, v in pyr_coeffs.items()}

    # Initialize normalizers
    normalizers = {k: np.zeros_like(v) for k, v in pyr_coeffs.items()}

    # Add high-pass normalizer
    if "residual_highpass" in pyr_coeffs:
        normalizers["residual_highpass"] = energies["residual_highpass"]

    # Compute normalizers across scales and orientations
    num_levels = (len(pyr_coeffs) - 2) // num_orientations  # Exclude high/low residuals
    for level in range(num_levels):
        for orientation in range(num_orientations):
            key = (level, orientation)
            if key in pyr_coeffs:
                normalizers[key] += energies[key]

    # Normalize each bandpass energy band
    normalized_coeffs = {}
    for level in range(num_levels):
        for orientation in range(num_orientations):
            key = (level, orientation)
            if key in pyr_coeffs:
                norm_value = (
                    normalizers.get((level - 1, orientation), 0) +
                    normalizers.get((level, orientation), 0) +
                    normalizers.get((level + 1, orientation), 0)
                ) + sigma**2  # Prevent divide by zero

                normalized_coeffs[key] = energies[key] / norm_value

    # Add low-pass and high-pass residuals without modification
    if "residual_highpass" in pyr_coeffs:
        normalized_coeffs["residual_highpass"] = pyr_coeffs["residual_highpass"]

    if "residual_lowpass" in pyr_coeffs:
        normalized_coeffs["residual_lowpass"] = pyr_coeffs["residual_lowpass"]

    return normalized_coeffs

In [7]:
import numpy as np
import cv2
import h5py
import pyrtools as pt
import os
import scipy.io as sio
import matplotlib.pyplot as plt
import json
import time


# File path (Modify according to your Google Drive structure)
hdf5_file = "G:/.shortcut-targets-by-id/1iPft89M8JBdXa7g3K5MBL0oLkoXgu5HL/V2_drift/nsd_raw/nsddata_stimuli/stimuli/nsd/nsd_stimuli.hdf5"

class NSDStimulusProcessor:
    def height(self, dims, bandwidth, allow_high_levels=False):
        """Calculate the number of spatial frequency levels for the steerable pyramid."""
        max_ht = int(np.log2(min(dims)))  # Compute max possible levels
        num_levels = int(np.log2(min(dims)) - np.log2(bandwidth))  # Default calculation

        if allow_high_levels:
            return min(num_levels, max_ht)  # Allow as many levels as possible
        return min(num_levels, 7)  # Default to 7 if allow_high_levels=False

    def __init__(self, stim_filename, output_folder="pyramid_expand/", max_images=5, allow_high_levels=False):
        """Initialize the processor with required parameters."""
        self.stim_filename = stim_filename
        self.output_folder = output_folder
        self.max_images = max_images
        self.allow_high_levels = allow_high_levels
        self.interp_img_size = 724
        self.background_size = 1024
        self.img_scaling = 0.5
        self.num_orientations = 8
        self.bandwidth = 1
        self.dims = [512, 512]  # Final downsampled dimensions
        os.makedirs(output_folder, exist_ok=True)

        # Open HDF5 file and get dataset shape **without loading images**
        with h5py.File(self.stim_filename, "r") as f:
            dataset_shape = f["imgBrick"].shape  # (73000, 425, 425, 3)
            self.total_images = dataset_shape[0]
            self.img_size_x, self.img_size_y, _ = dataset_shape[1:]

        self.num_imgs = min(self.max_images, self.total_images)
        print(f"Dataset contains {self.total_images} images. Loading only {self.num_imgs}.")

    def load_image(self, img_num):
        """Load a single image from HDF5 file on demand to save memory."""
        if img_num >= self.total_images:
            raise IndexError(f"Image index {img_num} is out of range (0-{self.total_images-1}).")

        with h5py.File(self.stim_filename, "r") as f:
            orig_img = f["imgBrick"][img_num]  # Load only one image at a time

        orig_img = np.array(orig_img, dtype=np.float32)  # Convert to float
        return orig_img  # Shape: (425, 425, 3)

    def interpolate_image(self, img):
        """Rescale image to `interp_img_size` while preserving details."""
        xq, yq = np.meshgrid(np.linspace(0, self.img_size_x - 1, self.interp_img_size),
                             np.linspace(0, self.img_size_y - 1, self.interp_img_size))
        interp_img = np.zeros((self.interp_img_size, self.interp_img_size, 3))
        for i in range(3):
            interp_img[:, :, i] = cv2.remap(img[:, :, i], xq.astype(np.float32), yq.astype(np.float32), cv2.INTER_LINEAR)
        return interp_img

    def add_fixation_point(self, img):
        """Add a semi-transparent red fixation point at the center."""
        fixation_radius = 8
        fixation_color = (255, 0, 0)  # Red
        overlay = img.copy()
        cv2.circle(overlay, (self.interp_img_size//2, self.interp_img_size//2), fixation_radius, fixation_color, -1)
        return cv2.addWeighted(overlay, 0.5, img, 0.5, 0)

    def pad_background(self, img):
        """Pad image to `background_size` with gray background."""
        big_img = np.full((self.background_size, self.background_size, 3), 127, dtype=np.uint8)
        pad_start = (self.background_size - self.interp_img_size) // 2
        big_img[pad_start:pad_start+self.interp_img_size, pad_start:pad_start+self.interp_img_size, :] = img
        return big_img

    def apply_steerable_pyramid(self, img):
      """Apply steerable pyramid transformation after resizing."""
      img_shape = img.shape[:2]  # Get final resized image dimensions
      self.num_levels = self.height(img_shape, self.bandwidth, self.allow_high_levels)  # Store levels

      print(f"Applying Steerable Pyramid with {self.num_levels} levels for {img_shape} image size.")

      # Convert image to float32 and normalize between -1 and 1
      img = img.astype(np.float32)
      img = (img - np.mean(img)) / np.std(img)  # Standardize image

      # Ensure grayscale shape (H, W)
      if len(img.shape) == 3:
          img = np.mean(img, axis=2)  # Convert to grayscale if necessary
          print(f"Converted to grayscale, new shape: {img.shape}")

      # Ensure input shape matches what the pyramid expects
      if img.shape != (512, 512):
          print(f"Warning: Unexpected image shape {img.shape} before pyramid.")

      return pt.pyramids.SteerablePyramidFreq(img, height=self.num_levels, order=self.num_orientations)



    def process_image(self, img_num, norm_resp=False):
      """Full pipeline: load, process, apply pyramid filters, and save."""
      start_time = time.time()  # Track processing time

      # MATLAB-style 1-based indexing for filenames
      pyramid_filename = f"pyrImg{img_num}.mat"
      save_path = os.path.join(self.output_folder, pyramid_filename)
      if os.path.exists(save_path):
          print(f"Image {img_num} already processed. Skipping...")
          return  # Skip processing

      print(f"Processing Image {img_num}/{self.num_imgs}")

      try:
          # Load and preprocess image
          img = self.load_image(img_num)
          img = self.interpolate_image(img)
          img = self.add_fixation_point(img)
          img = self.pad_background(img)

          img_gray = np.mean(img, axis=2).astype(np.float32)  # Convert to grayscale
          img_gray = (img_gray - np.mean(img_gray)) / np.std(img_gray)  # Normalize

          # Downsample
          img_resized = cv2.resize(img_gray, (self.dims[1], self.dims[0]), interpolation=cv2.INTER_AREA)

          # Apply Steerable Pyramid
          print(f"Applying Steerable Pyramid to Image {img_num}")
          pyr = self.apply_steerable_pyramid(img_resized)
          print(f"Steerable Pyramid successfully applied to Image {img_num}")

          # Apply Contrast Normalization (if enabled)
          if norm_resp:
              pyr_coeffs = norm_energies(pyr.pyr_coeffs, self.num_orientations, sigma=0.1)
              print("Applied normalized responses.")
          else:
              pyr_coeffs = pyr.pyr_coeffs

          print(f"Extracting frequency responses for Image {img_num}")
          sum_ori = [np.zeros_like(img_resized, dtype=np.float32) for _ in range(self.num_levels + 2)]
          model_ori = [np.zeros((self.num_orientations, *img_resized.shape), dtype=np.float32) for _ in range(self.num_levels + 2)]

          print(f"Pyramid has {len(pyr_coeffs)} coefficients: {list(pyr_coeffs.keys())}")

          # Process Each Level and Orientation
          for ilev in range(self.num_levels):
              for orientation in range(self.num_orientations):
                  key = (ilev, orientation)
                  if key in pyr_coeffs:
                      this_band = np.abs(pyr_coeffs[key])
                      this_band_resized = cv2.resize(this_band, (self.dims[1], self.dims[0]), interpolation=cv2.INTER_LINEAR)

                      sum_ori[ilev] += this_band_resized
                      model_ori[ilev][orientation] = this_band_resized
                  else:
                      print(f"Warning: Missing pyramid key {key}")

          # Remove extra orientations (ignore 8th orientation)
          # Extract only valid coefficients: 7 levels × 8 orientations
          pyr_coeffs = {
              k: v for k, v in pyr.pyr_coeffs.items()
              if (isinstance(k, tuple) and k[1] < 8)  # Keep only 8 orientations per level
              or k in ["residual_highpass", "residual_lowpass"]  # Keep low/high pass
          }
          # Add Low-Frequency Band
          if "residual_lowpass" in pyr_coeffs:
              lowpass = np.abs(pyr_coeffs["residual_lowpass"])
              lowpass_resized = cv2.resize(lowpass, (self.dims[1], self.dims[0]), interpolation=cv2.INTER_LINEAR)
              sum_ori[self.num_levels] = lowpass_resized
              model_ori[self.num_levels] = lowpass_resized
          else:
              print("Warning: Low-pass residual missing.")

          # Add High-Frequency Band
          if "residual_highpass" in pyr_coeffs:
              highpass = np.abs(pyr_coeffs["residual_highpass"])
              highpass_resized = cv2.resize(highpass, (self.dims[1], self.dims[0]), interpolation=cv2.INTER_LINEAR)
              sum_ori[self.num_levels+1] = highpass_resized
              model_ori[self.num_levels+1] = highpass_resized
          else:
              print("Warning: High-pass residual missing.")

          print(f"Extracting frequency responses for Image {img_num}")
          print(f"Pyramid has {len(pyr_coeffs)} coefficients: {list(pyr_coeffs.keys())}")


          # Save to .mat file
          sio.savemat(save_path, {
              "interpImgSize": np.array(self.interp_img_size, dtype=np.int32),
              "backgroundSize": np.array(self.background_size, dtype=np.int32),
              "imgScaling": np.array(self.img_scaling, dtype=np.float32),
              "numOrientations": np.array(self.num_orientations, dtype=np.int32),
              "bandwidth": np.array(self.bandwidth, dtype=np.int32),
              "dims": np.array(self.dims, dtype=np.int32),
              "bigImg": img_gray.astype(np.float32),  # Ensure correct dtype
              "sumOri": [s.astype(np.float32) for s in sum_ori],  # Keep as a list
              "modelOri": [m.astype(np.float32) for m in model_ori],  # Keep as a list
              "numLevels": np.array(self.num_levels, dtype=np.int32),
              "normResp": np.array(norm_resp, dtype=np.int8)  # Save normalization status
          })

          print(f"Saved: {save_path} (Time taken: {time.time() - start_time:.2f} sec)")

      except Exception as e:
          print(f"ERROR processing image {img_num}: {e}")



    def visualize_pyramid(self, img_num):
      """Visualize the original image, preprocessed image, and steerable pyramid decomposition."""

      # Check if img_num is within the loaded subset
      if img_num >= self.num_imgs:
          print(f"Error: Image index {img_num} is out of range (0-{self.num_imgs-1})")
          return

      print(f"Visualizing Steerable Pyramid for Image {img_num}")

      # Load the original image
      img = self.load_image(img_num)

      # Preprocess the image (interpolation, grayscale, etc.)
      img_interpolated = self.interpolate_image(img)
      img_with_fixation = self.add_fixation_point(img_interpolated)
      img_with_padding = self.pad_background(img_with_fixation)
      img_gray = np.mean(img_with_padding, axis=2)

      # Step 5: Downsample for steerable pyramid
      img_resized = cv2.resize(img_gray, (self.dims[1], self.dims[0]), interpolation=cv2.INTER_AREA)

      # Apply the Steerable Pyramid
      pyr = self.apply_steerable_pyramid(img_resized)

      # Visualization
      fig = plt.figure(figsize=(12, 7))

      # Visualize the original image
      ax1 = fig.add_subplot(2, 1, 1)
      ax1.imshow(img / 255.0)  # Normalize to [0, 1] for display
      ax1.set_title("Original Image")
      ax1.axis("off")

      # Visualize the preprocessed image
      ax2 = fig.add_subplot(2, 1, 2)
      ax2.imshow(img_resized, cmap="gray")
      ax2.set_title("Preprocessed Image")
      ax2.axis("off")

      plt.tight_layout()
      plt.show()

      # Subplots for Steerable Pyramid
      num_levels = pyr.num_scales
      num_orientations = self.num_orientations

      # Subplot for regular pyramid filters
      fig_filters, axes_filters = plt.subplots(num_levels, num_orientations, figsize=((20, 15)))
      fig_filters.suptitle(f"Steerable Pyramid - Regular Filters (Image {img_num})", fontsize=20)

      for ilev in range(num_levels):
          for orientation in range(num_orientations):
              key = (ilev, orientation)
              ax = axes_filters[ilev, orientation]
              if key in pyr.pyr_coeffs:
                  band = np.abs(pyr.pyr_coeffs[key])  # Take absolute value for display
                  ax.imshow(band, cmap="gray")
                  ax.set_title(f"Level {ilev+1}, Ori {orientation+1}", fontsize=8)
              else:
                  ax.axis("off")
              ax.set_xticks([])
              ax.set_yticks([])

      plt.tight_layout()
      plt.show()

      # Subplot for high-pass and low-pass residuals
      fig_residuals, axes_residuals = plt.subplots(1, 2, figsize=(15, 5))
      fig_residuals.suptitle(f"Steerable Pyramid - High/Low-Pass Residuals (Image {img_num})", fontsize=20)

      # High-pass residual
      if "residual_highpass" in pyr.pyr_coeffs:
          ax_hp = axes_residuals[0]
          ax_hp.imshow(np.abs(pyr.pyr_coeffs["residual_highpass"]), cmap="gray")
          ax_hp.set_title("High-Pass Residual", fontsize=14)
          ax_hp.axis("off")

      # Low-pass residual
      if "residual_lowpass" in pyr.pyr_coeffs:
          ax_lp = axes_residuals[1]
          ax_lp.imshow(np.abs(pyr.pyr_coeffs["residual_lowpass"]), cmap="gray")
          ax_lp.set_title("Low-Pass Residual", fontsize=14)
          ax_lp.axis("off")

      # Adjust layout
      plt.tight_layout()
      plt.show()

    def visualize_pyramid_with_fixation(self, img_num):
      """Visualize the original image, preprocessed image, and zoom in on the fixation point."""

      # Check if img_num is within the loaded subset
      if img_num >= self.num_imgs:
          print(f"Error: Image index {img_num} is out of range (0-{self.num_imgs-1})")
          return

      print(f"Visualizing Preprocessing Steps and Fixation Point for Image {img_num}")

      # Load and preprocess the image
      img = self.load_image(img_num)
      img_interpolated = self.interpolate_image(img)
      img_with_fixation = self.add_fixation_point(img_interpolated)
      img_with_padding = self.pad_background(img_with_fixation)

      # Visualization
      fig, axes = plt.subplots(1, 3, figsize=(18, 6))
      fig.suptitle(f"Preprocessing Steps for Image {img_num} and Fixation Point", fontsize=16)

      # Full preprocessed image with fixation
      axes[0].imshow(img_with_padding / 255.0)  # Normalize to [0, 1]
      axes[0].set_title("Preprocessed Image with Fixation")
      axes[0].axis("off")

      # Zoom into the fixation area
      fixation_radius = 20  # Define the region to zoom in (e.g., 20x20 pixels)
      center = self.background_size // 2
      zoom_img = img_with_padding[center-fixation_radius:center+fixation_radius,
                                  center-fixation_radius:center+fixation_radius, :]
      axes[1].imshow(zoom_img / 255.0)  # Normalize for display
      axes[1].set_title("Zoomed-in Fixation Area")
      axes[1].axis("off")

      # Show the red channel only
      red_channel = zoom_img[:, :, 0]
      axes[2].imshow(red_channel, cmap="Reds")  # Use "Reds" colormap to highlight red intensity
      axes[2].set_title("Red Channel of Fixation Area")
      axes[2].axis("off")

      plt.tight_layout()
      plt.show()


    def process_batch(self, start_idx, end_idx):
        """Process a batch of images while tracking progress."""
        batch_start_time = time.time()
        log_file = os.path.join(self.output_folder, "progress_log.json")

        # Ensure the log file is created if it doesn't exist
        if not os.path.exists(log_file):
            print("Creating new progress log file.")
            progress_data = {}
            with open(log_file, "w") as f:
                json.dump(progress_data, f, indent=4)
        else:
            try:
                with open(log_file, "r") as f:
                    progress_data = json.load(f)
            except json.JSONDecodeError:
                print("Warning: Log file corrupted. Resetting progress tracking.")
                progress_data = {}
                with open(log_file, "w") as f:
                    json.dump(progress_data, f, indent=4)

        print(f"Processing batch from {start_idx} to {end_idx - 1}")

        for img_num in range(start_idx, end_idx):
            try:
                pyramid_filename = f"pyrImg{img_num}.mat"
                save_path = os.path.join(self.output_folder, pyramid_filename)
                if str(img_num) in progress_data and os.path.exists(save_path):
                    print(f"Skipping Image {img_num} (already processed)")
                    continue

                self.process_image(img_num)
                progress_data[str(img_num)] = "processed"

                # Save the log file after each processed image
                with open(log_file, "w") as f:
                    json.dump(progress_data, f, indent=4)

            except Exception as e:
                print(f"Error processing image {img_num}: {e}. Skipping...")
                continue

        print(f"Batch {start_idx} - {end_idx} completed in {time.time() - batch_start_time:.2f} sec.")
        

    def process_all_in_batches(self, batch_size=10000):
        """Process all images in batches while skipping already processed images."""
        total_start_time = time.time()
        log_file = os.path.join(self.output_folder, "progress_log.json")

        # Ensure log file exists before processing
        if not os.path.exists(log_file):
            print("Creating new progress log file.")
            progress_data = {}
            with open(log_file, "w") as f:
                json.dump(progress_data, f, indent=4)
        else:
            try:
                with open(log_file, "r") as f:
                    progress_data = json.load(f)
            except json.JSONDecodeError:
                print("Warning: Log file corrupted. Resetting progress tracking.")
                progress_data = {}
                with open(log_file, "w") as f:
                    json.dump(progress_data, f, indent=4)

        total_batches = (self.total_images + batch_size - 1) // batch_size
        print(f"Total images: {self.total_images}. Processing in {total_batches} batches of {batch_size} images.")

        for batch_idx in range(total_batches):
            start_idx = batch_idx * batch_size
            end_idx = min((batch_idx + 1) * batch_size, self.total_images)

            # Skip fully processed batches
            if all(str(img) in progress_data and os.path.exists(os.path.join(self.output_folder, f"pyrImg{img}.mat")) for img in range(start_idx, end_idx)):
                print(f"Skipping batch {batch_idx} (already processed)")
                continue

            self.process_batch(start_idx, end_idx)

            # Save progress to log file after each batch
            with open(log_file, "w") as f:
                json.dump(progress_data, f, indent=4)

        print(f"Finished processing all images in {time.time() - total_start_time:.2f} sec.")


def visualize_image_with_check(processor, img_num):
      """Visualize and check dimensions of the processed image."""
      img = processor.load_image(img_num)
      print(f"Original Image Shape: {img.shape}")

      # Interpolation
      img_interpolated = processor.interpolate_image(img)
      print(f"Interpolated Image Shape: {img_interpolated.shape}")

      # Add fixation point
      img_with_fixation = processor.add_fixation_point(img_interpolated)

      # Pad with gray background
      img_padded = processor.pad_background(img_with_fixation)
      print(f"Padded Image Shape: {img_padded.shape}")

      # Grayscale Conversion
      img_gray = np.mean(img_padded, axis=2)
      print(f"Grayscale Image Shape: {img_gray.shape}")

      # Downsample to 512x512
      img_downsampled = cv2.resize(img_gray, (processor.dims[1], processor.dims[0]), interpolation=cv2.INTER_AREA)
      print(f"Downsampled Image Shape: {img_downsampled.shape}")

      # Display
      fig, axs = plt.subplots(1, 3, figsize=(15, 5))
      axs[0].imshow(img_with_fixation / 255.0)  # Normalize for display
      axs[0].set_title("With Fixation")
      axs[0].axis("off")

      axs[1].imshow(img_padded / 255.0)  # Normalize for display
      axs[1].set_title("Padded Image")
      axs[1].axis("off")

      axs[2].imshow(img_downsampled, cmap="gray")
      axs[2].set_title("Downsampled Image")
      axs[2].axis("off")

      plt.show()

if __name__ == "__main__":
    # Initialize the processor for all 73k images
    output_folder = "D:/NSD/pyramid_expand/"  # Change to your preferred path
    processor = NSDStimulusProcessor(hdf5_file, output_folder=output_folder, max_images=73000)

    # Process all stimuli in batches of 10k
    processor.process_all_in_batches(batch_size=73000)
    #processor.process_image(0)  # Replace 3 with any image number
    #visualize_image_with_check(processor, img_num=3)
    #processor.visualize_pyramid(3)





'''
If an error occurs (e.g., session crashes after processing 5,000 out of 10,000 images in a batch):

Re-run the batch:
processor.process_batch(start_idx=0, end_idx=10000)
The code will skip the first 5,000 images (already saved) and resume processing from image 5,001.
'''

FileNotFoundError: [Errno 2] Unable to synchronously open file (unable to open file: name = 'G:/.shortcut-targets-by-id/1iPft89M8JBdXa7g3K5MBL0oLkoXgu5HL/V2_drift/nsd_raw/nsddata_stimuli/stimuli/nsd/nsd_stimuli.hdf5', errno = 2, error message = 'No such file or directory', flags = 0, o_flags = 0)

In [ ]:
import json
with open("progress_log.json", "r") as f:
    progress = json.load(f)
print(f"Processed {len(progress)} images so far")


In [ ]:
import scipy.io as sio
import matplotlib.pyplot as plt
import numpy as np

# Load the .mat file
mat_file_path = "/content/drive/My Drive/Uni/msc/Zvi Lab/nsd/V1_Drift/pyrImg2.mat"
data = sio.loadmat(mat_file_path)

# Print available keys (variables inside the .mat file)
print("🔍 Keys in the .mat file:", data.keys())

# Inspect an example variable
print("🔹 interpImgSize:", data["interpImgSize"])
print("🔹 bigImg shape:", data["bigImg"].shape)
print("🔹 sumOri shape:", len(data["sumOri"]))
print("🔹 modelOri shape:", len(data["modelOri"]))


# Load the .mat file
data = sio.loadmat(mat_file_path)

# Extract and visualize the grayscale image
bigImg = data["bigImg"]

plt.figure(figsize=(6,6))
plt.imshow(bigImg, cmap="gray")
plt.title("Loaded Image from pyrImg3.mat")
plt.axis("off")
plt.show()

# Extract and visualize the first orientation of the first level
sumOri = data["sumOri"]
modelOri = data["modelOri"]

plt.figure(figsize=(10,5))

plt.subplot(1, 2, 2)
plt.imshow(np.array(modelOri[0][0][0]), cmap="gray")  # Choose the first orientation at the first level
plt.title("First Orientation at Level 1")
plt.axis("off")
plt.show()




# pRF
samples energy outputs from the steerable pyramid model using population receptive field (pRF) data for specific brain regions (e.g., V1, V2) and saves the results

In [ ]:
import os
import numpy as np
import nibabel as nib  # For .nii file reading
import scipy.io as sio
import h5py
import gc
from joblib import Parallel, delayed
from tqdm import tqdm
import tempfile, shutil



# Define paths
nsd_folder = r"D:\NSD"
pyramid_folder = r"D:\NSD\pyramid_expand"
prf_folder = r"C:\nsd\prfsample_expand"

# Set parameters
interp_img_size = 714
background_size = 1024
img_scaling = 0.5
num_levels = 7
num_orientations = 8
pix_per_deg = img_scaling * 714 / 8.4
deg_per_pix = 8.4 / (714 * img_scaling)

roi_names = ["V1v", "V1d", "V2v", "V2d", "V3v", "V3d", "hV4"]
test_mode = False  # Debugging mode - set to True for limited processing

# Function to load NIfTI files
def load_nifti(filepath):
    """Load NIfTI file if it exists, otherwise return None."""
    return nib.load(filepath).get_fdata() if os.path.exists(filepath) else None

# Function to load processed images log
def load_processed_images_log(log_path):
    """Load processed image numbers from a log file if it exists."""
    if os.path.exists(log_path):
        with open(log_path, "r") as log_file:
            return set(map(int, filter(str.isdigit, log_file.read().splitlines())))
    return set()

def process_single_image(imgNum, roiGaus, num_levels, num_orientations):
    pyramid_filename = os.path.join(pyramid_folder, f"pyrImg{imgNum - 1}.mat")
    try:
        data = sio.loadmat(pyramid_filename)
        sumOri = data["sumOri"]
        modelOri = data["modelOri"][0]
    except Exception as e:
        print(f"Failed to load image {imgNum}: {e}")
        return imgNum, {}, {}

    prfSampleLev_partial = {}
    prfSampleLevOri_partial = {}

    for roinum, G_all in roiGaus.items():
        n_voxels = G_all.shape[0]
        prf_levels = np.zeros((n_voxels, num_levels + 2), dtype=np.float32)  # Use float32 for lower memory usage
        prf_oris = np.zeros((n_voxels, num_levels + 2, num_orientations), dtype=np.float32)

        for ilev in range(min(num_levels + 2, len(sumOri))):
            prf_levels[:, ilev] = np.tensordot(G_all, sumOri[ilev], axes=([1, 2], [0, 1]))

        for ilev in range(min(num_levels, len(modelOri))):
            for iori in range(num_orientations):
                prf_oris[:, ilev, iori] = np.sum(modelOri[ilev][iori] * G_all, axis=(1, 2))

        prfSampleLev_partial[roinum] = prf_levels
        prfSampleLevOri_partial[roinum] = prf_oris

    return imgNum, prfSampleLev_partial, prfSampleLevOri_partial

def prf_sample_model_expand(isub, visualRegion, batch_size=100):
    print(f"\n-- Running pRF Sampling for Subject {isub}, Region {visualRegion} ---")

    h5_filename = os.path.join(prf_folder, f"prfSampleStim_v{visualRegion}_sub{isub}.h5")
    log_path = os.path.join(prf_folder, f"processed_images_sub{isub}.log")
    processed_images = load_processed_images_log(log_path)

    nsdDesignPath = os.path.join(nsd_folder, "experiments", "nsd_expdesign.mat")
    if not os.path.exists(nsdDesignPath):
        print(f"ERROR: NSD Design file not found: {nsdDesignPath}")
        return

    nsdDesign = sio.loadmat(nsdDesignPath)
    subjectim = nsdDesign["subjectim"]
    masterordering = nsdDesign["masterordering"].flatten() - 1
    valid_masterordering = masterordering[masterordering < subjectim.shape[1]]
    allImgs = np.unique(subjectim[isub - 1, valid_masterordering])

    roi_map = {1: [0, 1], 2: [2, 3], 3: [4, 5], 4: [6]}
    rois = roi_map.get(visualRegion, [])

    if test_mode:
        allImgs = allImgs[:5]

    print(f"Total images to process: {len(allImgs)}")

    betas_folder = os.path.join(nsd_folder, f"subj{isub:02d}", "func1pt8mm")
    prf_files = ["prf_angle.nii.gz", "prf_eccentricity.nii.gz", "prf_size.nii.gz", "R2.nii.gz"]
    angData, eccData, sizeData, r2Data = [load_nifti(os.path.join(betas_folder, f)) for f in prf_files]
    visRoiData = load_nifti(os.path.join(betas_folder, "roi", "prf-visualrois.nii.gz"))

    if any(data is None for data in [angData, eccData, sizeData, r2Data, visRoiData]):
        print("ERROR: Missing pRF data. Check file paths.")
        return

    prfSampleLev, prfSampleLevOri, roiPrf = {}, {}, {}
    gauss_file = os.path.join(prf_folder, f"roiGaussians_sub{isub}_v{visualRegion}.h5")
    roiGaus = {}
    missing_rois = []

    if os.path.exists(gauss_file):
        print(f"Loading cached Gaussians from {gauss_file}")
        with h5py.File(gauss_file, "r") as f:
            for roinum in roi_map.get(visualRegion, []):
                key = f"roi_{roinum}"
                if key in f:
                    roiGaus[roinum] = f[key][:]
                    # Create dummy prf containers - need this part because the cached Gaussians don’t include ecc, ang, or the number of voxels
                    roi_mask = visRoiData == (roinum + 1)
                    ecc = eccData[roi_mask]
                    ang = angData[roi_mask]
                    sz = sizeData[roi_mask] * pix_per_deg
                    x0 = ecc * np.cos(np.deg2rad(ang) * 2)
                    y0 = ecc * np.sin(np.deg2rad(ang) * 2)

                    roiPrf[roinum] = {"ecc": ecc, "ang": ang, "sz": sz, "x": x0, "y": y0}
                    prfSampleLev[roinum] = np.zeros((len(allImgs), len(ecc), num_levels + 2))
                    prfSampleLevOri[roinum] = np.zeros((len(allImgs), len(ecc), num_levels + 2, num_orientations))
                else:
                    missing_rois.append(roinum)
    else:
        # If the file doesn't exist at all, compute everything
        missing_rois = roi_map.get(visualRegion, [])

    if missing_rois:
        print(f"Computing Gaussians for missing ROIs: {missing_rois}")
        X, Y = np.meshgrid(
            np.linspace(-background_size * img_scaling / 2, background_size * img_scaling / 2, 512),
            np.linspace(-background_size * img_scaling / 2, background_size * img_scaling / 2, 512)
        )
        with h5py.File(gauss_file, "a") as f:  # append mode
            for roinum in missing_rois:
                roi_mask = visRoiData == (roinum + 1)
                if np.sum(roi_mask) == 0:
                    print(f"Skipping empty ROI: {roinum}")
                    continue

                ecc = eccData[roi_mask]
                ang = angData[roi_mask]
                sz = sizeData[roi_mask] * pix_per_deg
                x0 = ecc * np.cos(np.deg2rad(ang) * 2)
                y0 = ecc * np.sin(np.deg2rad(ang) * 2)

                roiPrf[roinum] = {"ecc": ecc, "ang": ang, "sz": sz, "x": x0, "y": y0}
                prfSampleLev[roinum] = np.zeros((len(allImgs), len(ecc), num_levels + 2))
                prfSampleLevOri[roinum] = np.zeros((len(allImgs), len(ecc), num_levels + 2, num_orientations))

                X_diff = X.reshape(1, 512, 512) - x0[:, None, None]
                Y_diff = Y.reshape(1, 512, 512) - y0[:, None, None]
                G_all = np.exp(-(X_diff**2 + Y_diff**2) / (2 * sz[:, None, None]**2))


                roiGaus[roinum] = G_all
                f.create_dataset(f"roi_{roinum}", data=G_all, compression="gzip")

        print(f"Updated Gaussian cache with missing ROIs")



    work_list = []
    image_index_map = {}
    kept_imgs = []


    for iimg, imgNum in enumerate(allImgs):
        if imgNum in processed_images:
            print(f"Skipping already processed image {imgNum}")
            continue

        pyramid_filename = os.path.join(pyramid_folder, f"pyrImg{iimg}.mat")
        if not os.path.exists(pyramid_filename):
            print(f" Missing pyramid file: {pyramid_filename}")
            continue

        # Only pass image numbers now, not the full data
        work_list.append(imgNum)
        image_index_map[imgNum] = len(kept_imgs)
        kept_imgs.append(imgNum)

    if not work_list:
        print("\n No new images to process. Skipping save.")
        return

    print(f"\nProcessing {len(work_list)} new images")

    num_batches = len(work_list) // batch_size + (1 if len(work_list) % batch_size != 0 else 0)

    
    for batch_num in range(num_batches):
        batch = work_list[batch_num * batch_size:(batch_num + 1) * batch_size]
        print(f"Processing batch {batch_num + 1}/{num_batches} with {len(batch)} images")

        # Wrap the parallel call with tqdm to show progress
        if len(batch) == 0:
            print(f"Skipping empty batch {batch_num + 1}/{num_batches}")
            continue

        results = Parallel(n_jobs=5)(
            delayed(process_single_image)(imgNum, roiGaus, num_levels, num_orientations)
            for imgNum in tqdm(batch, desc=f"Batch {batch_num + 1}/{num_batches}", unit="image", ncols=100)
        )

        for imgNum, partial_lev, partial_ori in results:
            iimg = image_index_map[imgNum]
            for roinum in partial_lev:
                prfSampleLev[roinum][iimg, :, :] = partial_lev[roinum]
                prfSampleLevOri[roinum][iimg, :, :, :] = partial_ori[roinum]

            #free memory after each image
            del partial_lev, partial_ori
            gc.collect()

        # Use a temp file to avoid corruption
        with tempfile.NamedTemporaryFile(delete=False, suffix=".h5", dir=prf_folder) as tmp_file:
            tmp_filename = tmp_file.name

        with h5py.File(tmp_filename, "w") as f:
            for roinum in prfSampleLev:
                f.create_dataset(f"prfSampleLev/roi_{roinum}", data=prfSampleLev[roinum], compression="gzip")
                f.create_dataset(f"prfSampleLevOri/roi_{roinum}", data=prfSampleLevOri[roinum], compression="gzip")

            f.create_dataset("allImgs", data=np.array(kept_imgs, dtype=np.int32))
            f.create_dataset("rois", data=np.array(rois, dtype=np.int32))
            f.attrs["numLevels"] = num_levels
            f.attrs["numOrientations"] = num_orientations
            f.attrs["interpImgSize"] = interp_img_size
            f.attrs["backgroundSize"] = background_size
            f.attrs["pixPerDeg"] = pix_per_deg

        # Atomically move to final location (safe write)
        shutil.move(tmp_filename, h5_filename)

        print(f"Batch {batch_num + 1} processed and saved")

        # Free up memory after processing the batch
        del results  # Remove reference to the batch results
        gc.collect()  # Manually trigger garbage collection

        # Only log images that were not already logged
        new_logged_imgs = [str(img) for img in batch if img not in processed_images]

        if new_logged_imgs:
            with open(log_path, "a") as log_file:
                log_file.write("\n".join(new_logged_imgs) + "\n")
            print(f"Logged {len(new_logged_imgs)} new images")




if __name__ == "__main__":
    from multiprocessing import freeze_support
    freeze_support()  # optional but recommended on Windows
    prf_sample_model_expand(isub=1, visualRegion=1, batch_size=5)



-- Running pRF Sampling for Subject 1, Region 1 ---
Total images to process: 10000

Processing 10000 new images
Batch 1/2000 with 5 images


Batch 1/2000: 100%|██████████████████████████████████████████████████| 5/5 [00:00<00:00, 23.53img/s]


KeyboardInterrupt: 

Check prfSampleStim_v1_sub1.h5 is valid

In [ ]:
import h5py

# Path to your file
file_path = r"C:\nsd\prfsample_expand\prfSampleStim_v1_sub1.h5"

with h5py.File(file_path, 'r') as f:
    for key in f.keys():
        print("Top-level key:", key)
        try:
            print("  Subkeys:", list(f[key].keys()))
        except Exception as e:
            print("  Failed to read subkeys:", e)


Inspecting group: prfSampleLev


RuntimeError: Unable to get group info (addr overflow, addr = 2104, size = 328, eoa = 2048)

# Regression

In [ ]:
import numpy as np
import nibabel as nib
import scipy.io as sio
import os
from scipy.stats import zscore
from sklearn.linear_model import LinearRegression
from joblib import Parallel, delayed


# --------------------------------------------------
# Utility: Compute R² between residual and original
# --------------------------------------------------

def rsquared(Xresid, Xorig):
    return 1 - np.sum((Xresid) ** 2) / np.sum((Xorig - np.mean(Xorig)) ** 2)


# ----------------------------------------------------------------------------
# Main Function: Perform voxel-wise regression of PRF features on NSD betas
# isub: subject index (1-based like MATLAB)
# visual_regions: list of visual regions (e.g., [1, 2, 3])
# to_zscore: normalization method (0: none, 1: z-score, 2: mean, etc.)
# ----------------------------------------------------------------------------

def regress_prf_split_expand(isub, visual_regions=[1], to_zscore=0):
    # -------------------------------
    # Setup & File Paths
    # -------------------------------

    # Number of sessions per subject
    nsessions_sub = [40, 40, 32, 30, 40, 32, 40, 30]
    nsessions = nsessions_sub[isub - 1]
    nsplits = nsessions  # One split per session

    # bandpass info
    bandpass = 1
    band_min = 1
    band_max = 7

    # Set data folders
    boxfolder = f"/misc/data18/rothzn/nsd/prfsample_expand/"
    save_folder = f"/misc/data18/rothzn/nsd/repDrift_expand/"
    betas_folder = f"/misc/data18/rothzn/nsd/sub{isub}_betas_func1pt8mm/"
    nsd_folder = "/misc/data18/rothzn/nsd/"
    roifolder = f"/misc/data18/rothzn/nsd/sub{isub}_betas_func1pt8mm/"
    visual_rois_file = os.path.join(roifolder, "prf-visualrois.nii")

    # Load visual ROI mask (NIfTI)
    vis_roi_img = nib.load(visual_rois_file)
    vis_roi_data = vis_roi_img.get_fdata().flatten()  # Flatten for voxel indexing

    roi_names = ["V1v", "V1d", "V2v", "V2d", "V3v", "V3d", "hV4"]

    # Load NSD experiment design
    nsd_design = sio.loadmat(os.path.join(nsd_folder, "nsd_expdesign.mat"))
    master_ordering = nsd_design["masterordering"].squeeze() - 1  # MATLAB 1-indexed
    sub_design = nsd_design["subjectim"][isub - 1, master_ordering].squeeze()

    # ------------------------------------------
    # Split trial matrix (1 row per split)
    # ------------------------------------------

    ntrials = len(master_ordering)
    trials_per_session = 10 * 75  # Each session: 10 blocks × 75 trials
    split_img_trials = np.zeros((nsplits, ntrials), dtype=bool)

    itrial = 0
    for isplit in range(nsplits):  # Define split_img_trials once
        split_img_trials[isplit, itrial : itrial + trials_per_session] = True
        itrial += trials_per_session

    # ------------------------------------------
    # Loop over Visual Regions
    # ------------------------------------------

    for visual_region in visual_regions: # # Outer loop: per visual ROI (e.g., V1v, V1d...)
        print(f"Processing visual region: {visual_region}")

        # --- Load PRF sample features for this region ---
        prf_file = os.path.join(
            boxfolder, f"prfSampleStim_v{visual_region}_sub{isub}.mat")
        prf_data = sio.loadmat(prf_file)  # Load the PRF energy for this region

        prf_sample_lev_ori = prf_data["prfSampleLevOri"]  # Orientation × Level
        prf_sample_lev = prf_data['prfSampleLev'] # only lvl
        rois = prf_data['rois'].squeeze() # ROI labels (e.g 1-7)
        all_imgs = prf_data['allImgs'].squeeze()  # full image set (e.g 10k)
        num_levels = int(prf_data['numLevels'][0][0])
        num_orientations = int(prf_data['numOrientations'][0][0])
        roi_prf = prf_data["roiPrf"]  # PRF metadata (kept for save)

        # --- Init containers for each ROI ---
        roi_betas = {roinum: [] for roinum in range(len(rois))}  # beta responses
        roi_ind = {}  # voxel indices per ROI

        # --- Load and normalize beta data per session ---
        for isession in range(1, nsessions + 1):  # For each session
            beta_file = os.path.join(betas_folder, f"betas_session{isession:02d}.nii")
            beta_img = nib.load(beta_file)
            beta_data = beta_img.get_fdata().astype(np.float64)
            beta_data = beta_data / 300  # Convert to percent signal change
            beta_data = beta_data.reshape(-1, beta_data.shape[-1])  # Flatten 4D to 2D

            for roinum, iroi in enumerate(rois): # For each ROI within this visual region
                vox_idx = vis_roi_data == iroi
                roi_ind[roinum] = np.where(vox_idx)[0]
                roi_voxels = beta_data[vox_idx, :]  # Get betas for this ROI

                # --- Apply normalization method ---
                if to_zscore == 1:
                    norm_voxels = zscore(roi_voxels, axis=1, ddof=0)
                elif to_zscore == 2:
                    norm_voxels = roi_voxels - roi_voxels.mean(axis=1, keepdims=True)
                elif to_zscore == 3:
                    mean_ = roi_voxels.mean(axis=1, keepdims=True)
                    norm_voxels = mean_ + zscore(roi_voxels - mean_, axis=1, ddof=0)
                elif to_zscore == 4:
                    norm_voxels = roi_voxels - roi_voxels.mean()
                else:
                    norm_voxels = roi_voxels  # No normalization

                roi_betas[roinum].append(norm_voxels)

        # --- Concatenate sessions along time axis ---
        for roinum in roi_betas:
            roi_betas[roinum] = np.concatenate(roi_betas[roinum], axis=1)

        # --- Match presented images to full image set ---
        img_trials_mask, img_num = (
            np.isin(sub_design, all_imgs, assume_unique=True),
            None,
        )
        img_num = np.searchsorted(
            all_imgs, sub_design[img_trials_mask]
        )  # Shape: (n_trials,)

        # --- Ensure trial counts match for this subject ---
        for roinum in roi_betas:
            roi_betas[roinum] = roi_betas[roinum][:, : split_img_trials.shape[1]]
        split_img_trials = split_img_trials[:, : roi_betas[roinum].shape[1]]
        img_num = img_num[: roi_betas[roinum].shape[1]]

        max_num_trials = split_img_trials.sum(axis=1).max()  # Used for array size later

        # Prepare containers per ROI
        nsd = {}
        nvox = {}
        r2 = {}
        r2ori = {}
        r2split = {}
        r2ori_split = {}
        rss_split = {}
        rss_ori_split = {}
        pearson_r = {}
        pearson_rori = {}
        sess_betas = {}
        sess_std_betas = {}

        vox_coef = {}
        vox_ori_coef = {}
        vox_pred_ori_coef = {}
        vox_ori_pred_ori_coef = {}
        vox_resid_ori_coef = {}
        vox_ori_resid_ori_coef = {}

        vox_pred_ori_r2 = {}
        vox_resid_ori_r2 = {}
        vox_ori_pred_ori_r2 = {}
        vox_ori_resid_ori_r2 = {}

        vox_coef_corr = {}
        vox_ori_coef_corr = {}
        vox_coef_corr_with_const = {}
        vox_ori_coef_corr_with_const = {}

        # ------------------------------------------------------------
        # Voxel-wise Regression for Each ROI and Each Split
        # ------------------------------------------------------------
        for roinum in range(len(rois)):
            print(f"  → Processing ROI {roinum} (ID {rois[roinum]})")
            nvox[roinum] = roi_betas[roinum].shape[0]
            L = nvox[roinum]

            # Allocate arrays for this ROI
            shape_lev = num_levels + 1
            shape_ori = num_levels * num_orientations + 1

            # Initialize coefficient and residual matrices per voxel per split
            vox_coef[roinum] = np.zeros((nsplits, L, shape_lev))
            vox_ori_coef[roinum] = np.zeros((nsplits, L, shape_ori))
            vox_pred_ori_coef[roinum] = np.zeros((nsplits, L, shape_ori))
            vox_ori_pred_ori_coef[roinum] = np.zeros((nsplits, L, shape_ori))
            vox_resid_ori_coef[roinum] = np.zeros((nsplits, L, shape_ori))
            vox_ori_resid_ori_coef[roinum] = np.zeros((nsplits, L, shape_ori))

            vox_pred_ori_r2[roinum] = np.zeros((nsplits, L))
            vox_resid_ori_r2[roinum] = np.zeros((nsplits, L))
            vox_ori_pred_ori_r2[roinum] = np.zeros((nsplits, L))
            vox_ori_resid_ori_r2[roinum] = np.zeros((nsplits, L))
            r2[roinum] = np.zeros((nsplits, L))
            r2ori[roinum] = np.zeros((nsplits, L))
            sess_betas[roinum] = np.zeros((nsplits, L))
            sess_std_betas[roinum] = np.zeros((nsplits, L))

            # Loop over splits
            for isplit in range(nsplits):
                img_trials_split = split_img_trials[isplit]
                trial_idx = np.where(img_trials_split)[0]

                betas_split = roi_betas[roinum][:, trial_idx]  # shape: voxels × trials
                sess_betas[roinum][isplit] = np.mean(betas_split, axis=1)
                sess_std_betas[roinum][isplit] = np.std(betas_split, axis=1)

                for ivox in range(L):
                    # Voxel betas
                    y = betas_split[ivox, :]

                    # ---------- PRF: Basic Level Model ----------
                    X = prf_sample_lev[roinum][img_num[trial_idx], ivox, :].squeeze()
                    X = np.column_stack([X, np.ones(X.shape[0])])  # Add constant term

                    coef = np.linalg.lstsq(X, y, rcond=None)[0]
                    vox_coef[roinum][isplit, ivox, :] = coef
                    pred = np.dot(X, coef) # pred = X @ coef
                    residual = y - pred
                    r2[roinum][isplit, ivox] = rsquared(residual, y)

                    # ---------- PRF: Orientation-Level Model ----------
                    X_ori = prf_sample_lev_ori[roinum][
                        img_num[trial_idx], ivox, :, :
                    ].reshape(len(y), -1)
                    X_ori = np.column_stack(
                        [
                            X_ori,
                            X[:, -2],  # lowest SF
                            X[:, -3],  # highest SF
                            np.ones(X.shape[0]),  # constant
                        ]
                    )

                    coef_ori = np.linalg.lstsq(X_ori, y, rcond=None)[0]
                    vox_ori_coef[roinum][isplit, ivox, :] = coef_ori
                    pred_ori = np.dot(X_ori, coef_ori)  # pred_ori = X_ori @ coef_ori
                    residual_ori = y - pred_ori
                    r2ori[roinum][isplit, ivox] = rsquared(residual_ori, y)

                    # ---------- Residual regressions ----------
                    # Predict orientation residuals with orientation model
                    vox_ori_resid_ori_coef[roinum][isplit, ivox, :] = np.linalg.lstsq(
                        X_ori, residual_ori, rcond=None
                    )[0]
                    pred_ori_resid = np.dot(X_ori,vox_ori_resid_ori_coef[roinum][isplit, ivox, :])  # pred_ori_resid = X_ori @ vox_ori_resid_ori_coef[roinum][isplit, ivox, :]
                    vox_ori_resid_ori_r2[roinum][isplit, ivox] = rsquared(
                        residual_ori - pred_ori_resid, y
                    )

                    # Predict basic residuals with orientation model
                    vox_resid_ori_coef[roinum][isplit, ivox, :] = np.linalg.lstsq(
                        X_ori, residual, rcond=None
                    )[0]
                    pred_resid_ori =np.dot(X_ori, vox_resid_ori_coef[roinum][isplit, ivox, :]) # pred_resid_ori = X_ori @ vox_resid_ori_coef[roinum][isplit, ivox, :]
                    vox_resid_ori_r2[roinum][isplit, ivox] = rsquared(
                        residual - pred_resid_ori, y
                    )

                    # Predict prediction with orientation model
                    pred_from_basic = pred
                    vox_pred_ori_coef[roinum][isplit, ivox, :] = np.linalg.lstsq(
                        X_ori, pred_from_basic, rcond=None
                    )[0]
                    pred_pred_ori = np.dot(X_ori, vox_pred_ori_coef[roinum][isplit, ivox, :])
                    vox_pred_ori_r2[roinum][isplit, ivox] = rsquared(
                        pred_from_basic - pred_pred_ori, y
                    )

                    # Predict full orientation prediction with same
                    vox_ori_pred_ori_coef[roinum][isplit, ivox, :] = np.linalg.lstsq(
                        X_ori, pred_ori, rcond=None
                    )[0]
                    pred_ori_pred_ori = (
                        np.dot(X_ori, vox_ori_pred_ori_coef[roinum][isplit, ivox, :])
                    )
                    vox_ori_pred_ori_r2[roinum][isplit, ivox] = rsquared(
                        pred_ori - pred_ori_pred_ori, y
                    )
            # Containers for cross-split comparisons
            r2split[roinum] = np.zeros((nsplits, L, nsplits))
            r2ori_split[roinum] = np.zeros((nsplits, L, nsplits))
            rss_split[roinum] = np.zeros((nsplits, L, nsplits))
            rss_ori_split[roinum] = np.zeros((nsplits, L, nsplits))
            pearson_r[roinum] = np.zeros((nsplits, L, nsplits))
            pearson_rori[roinum] = np.zeros((nsplits, L, nsplits))

            # ----------------------------
            # Loop: isplit (target session)
            # ----------------------------
            for isplit in range(nsplits):
                trial_idx = np.where(split_img_trials[isplit])[0]
                y = roi_betas[roinum][:, trial_idx]  # shape: (voxels, trials)

                for ivox in range(L):
                    y_vox = y[ivox, :]

                    # Get design matrices for isplit
                    X = prf_sample_lev[roinum][img_num[trial_idx], ivox, :].squeeze()
                    X = np.column_stack([X, np.ones(X.shape[0])])

                    X_ori = prf_sample_lev_ori[roinum][
                        img_num[trial_idx], ivox, :, :
                    ].reshape(len(trial_idx), -1)
                    X_ori = np.column_stack(
                        [
                            X_ori,
                            X[:, -2],  # lowest SF
                            X[:, -3],  # highest SF
                            np.ones(X.shape[0]),
                        ]
                    )

                    # ----------------------------
                    # Loop: nextSplit (model source)
                    # ----------------------------
                    for next_split in range(nsplits):
                        coef = vox_coef[roinum][next_split, ivox, :]
                        coef_ori = vox_ori_coef[roinum][next_split, ivox, :]

                        # Predict from basic model
                        y_pred = np.dot(X, coef)  # y_pred = X @ coef
                        residual = y_vox - y_pred
                        r2split[roinum][isplit, ivox, next_split] = rsquared(
                            residual, y_vox
                        )
                        rss_split[roinum][isplit, ivox, next_split] = np.sum(
                            residual**2
                        )
                        pearson_r[roinum][isplit, ivox, next_split] = np.corrcoef(
                            y_vox, y_pred
                        )[0, 1]

                        # Predict from orientation model
                        y_pred_ori =np.dot(X_ori, coef_ori)  # y_pred_ori = X_ori @ coef_ori
                        residual_ori = y_vox - y_pred_ori
                        r2ori_split[roinum][isplit, ivox, next_split] = rsquared(
                            residual_ori, y_vox
                        )
                        rss_ori_split[roinum][isplit, ivox, next_split] = np.sum(
                            residual_ori**2
                        )
                        pearson_rori[roinum][isplit, ivox, next_split] = np.corrcoef(
                            y_vox, y_pred_ori
                        )[0, 1]

            # --------------------------------------------------------
            # PART 6A: Correlation Between Splits for Each Voxel
            # --------------------------------------------------------
            vox_coef_corr[roinum] = np.zeros((L, nsplits, nsplits))
            vox_ori_coef_corr[roinum] = np.zeros((L, nsplits, nsplits))
            vox_coef_corr_with_const[roinum] = np.zeros((L, nsplits, nsplits))
            vox_ori_coef_corr_with_const[roinum] = np.zeros((L, nsplits, nsplits))

            for ivox in range(L):
                # Without constant (drop last column)
                coef_mat = vox_coef[roinum][:, ivox, :-1]
                ori_mat = vox_ori_coef[roinum][:, ivox, :-1]

                vox_coef_corr[roinum][ivox] = np.corrcoef(coef_mat, rowvar=False)
                vox_ori_coef_corr[roinum][ivox] = np.corrcoef(ori_mat, rowvar=False)

                # With constant
                coef_mat_full = vox_coef[roinum][:, ivox, :]
                ori_mat_full = vox_ori_coef[roinum][:, ivox, :]

                vox_coef_corr_with_const[roinum][ivox] = np.corrcoef(
                    coef_mat_full, rowvar=False
                )
                vox_ori_coef_corr_with_const[roinum][ivox] = np.corrcoef(
                    ori_mat_full, rowvar=False
                )
        
        # -----------------------------------------------------
        # Package Results in Dictionary and Save
        # -----------------------------------------------------
        nsd = {
            "voxOriCoefCorrWithConst": vox_ori_coef_corr_with_const,
            "voxCoefCorrWithConst": vox_coef_corr_with_const,
            "voxOriCoefCorr": vox_ori_coef_corr,
            "voxCoefCorr": vox_coef_corr,
            "r2": r2,
            "r2ori": r2ori,
            "r2split": r2split,
            "r2oriSplit": r2ori_split,
            "rssOriSplit": rss_ori_split,
            "rssSplit": rss_split,
            "pearsonRori": pearson_rori,
            "pearsonR": pearson_r,
            "imgNum": img_num,
            "splitImgTrials": split_img_trials,
            "voxCoef": vox_coef,
            "voxOriCoef": vox_ori_coef,
            "voxPredOriCoef": vox_pred_ori_coef,
            "voxOriPredOriCoef": vox_ori_pred_ori_coef,
            "voxResidOriCoef": vox_resid_ori_coef,
            "voxOriResidOriCoef": vox_ori_resid_ori_coef,
            "voxPredOriR2": vox_pred_ori_r2,
            "voxResidOriR2": vox_resid_ori_r2,
            "voxOriPredOriR2": vox_ori_pred_ori_r2,
            "voxOriResidOriR2": vox_ori_resid_ori_r2,
            "sessBetas": sess_betas,
            "sessStdBetas": sess_std_betas,
            "roiInd": roi_ind,
        }

        # Choose filename suffix based on z-score method
        zscore_str = {
            1: "_zscore",
            2: "_zeroMean",
            3: "_equalStd",
            4: "_zeroROImean",
        }.get(to_zscore, "")

        # Save to .mat
        save_path = os.path.join(
            save_folder,
            f"regressPrfSplit_session_v{visual_region}_sub{isub}{zscore_str}.mat",
        )

        sio.savemat(
            save_path,
            {
                "nsd": nsd,
                "numLevels": num_levels,
                "numOrientations": num_orientations,
                "rois": rois,
                "nvox": nvox,
                "roiPrf": roi_prf,
                "nsplits": nsplits,
            },
        )

        print(f"Saved results to: {save_path}")